# PDF Parsing with GLM-OCR

Parses every PDF under `data/textbook/` with [zai-org/GLM-OCR](https://huggingface.co/zai-org/GLM-OCR) — a compact **0.9B** encoder-decoder OCR VLM (CogViT visual encoder + lightweight cross-modal connector + GLM-0.5B decoder, trained with Multi-Token Prediction + RL). Supports text/formula/table recognition and JSON information extraction.

Dramatically lighter than the other two parser notebooks (olmOCR-2-7B-1025-FP8 ~8GB, Infinity-Parser2-Pro-35B ~70GB): **0.9B in BF16 is under 2GB of weights** — runs on almost any CUDA GPU or Apple Silicon Mac, and is even usable on CPU for smoke-testing (slow, but not impractical given the size). Reported throughput on the model card: ~1.86 pages/sec (PDF) / ~0.67 images/sec on suitable hardware.
- CUDA box: any GPU with a few GB free VRAM.
- Apple Silicon: runs via PyTorch's MPS backend (`device="mps"`).
- CPU fallback also wired up here (unlike the other two notebooks) since 0.9B is small enough to be a genuine option, not just a smoke-test path.

Recommended deps (not in this project's `pyproject.toml` — install separately, e.g. into a `.venv-glm-ocr` venv):
```
uv venv .venv-glm-ocr --python 3.13
# CUDA box:
uv pip install --python .venv-glm-ocr \
    torch --index-url https://download.pytorch.org/whl/cu124 \
    transformers accelerate pillow pymupdf tqdm pyyaml
# Apple Silicon / CPU:
uv pip install --python .venv-glm-ocr \
    torch transformers accelerate pillow pymupdf tqdm pyyaml
```

GLM-OCR's card doesn't document a single "whole-page, preserve-layout, tables-as-HTML, formulas-as-LaTeX" prompt the way olmOCR does (its primary documented modes are per-task: `Text Recognition:`, `Formula Recognition:`, `Table Recognition:`, or strict-JSON information extraction via its official SDK + PP-DocLayoutV3 for layout). To keep this notebook's output contract identical to [olmocr.ipynb](olmocr.ipynb) (one Markdown file per PDF, page-broken, no bbox/crop step), we use a custom whole-page prompt modeled on olmOCR's, asking the model to transcribe the full page as Markdown with HTML tables and LaTeX equations in one shot — outside the model's officially benchmarked usage, so treat quality as unverified. It does **not** emit bboxes in this mode, so — like `olmocr.ipynb` and unlike `infinity_parser.ipynb` — there's no crop-visuals step; output is saved standalone under `output/<pdf_stem>/glm_ocr/`.

## Setup

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # notebook lives in notebooks/

TEXTBOOK_DIR = PROJECT_ROOT / "data" / "textbook"
OUTPUT_ROOT = PROJECT_ROOT / "output"

pdf_paths = sorted(TEXTBOOK_DIR.rglob("*.pdf"))
assert pdf_paths, f"No PDFs found under {TEXTBOOK_DIR}"

len(pdf_paths), pdf_paths[0]

## Render PDF pages to images

Rasterizes each PDF page to PNG via PyMuPDF, at 150 DPI. GLM-OCR's processor (`Glm46VImageProcessor`) handles its own internal resizing (patch-based, `shortest_edge`/`longest_edge` bounds in `preprocessor_config.json`), so — unlike `olmocr.ipynb`'s manual longest-side-1288px prescale for the Qwen2.5-VL processor — there's no need to prescale here; a fixed-DPI render matching `infinity_parser.ipynb`'s convention is enough.

In [ ]:
import fitz  # PyMuPDF

RENDER_DPI = 150


def render_pages(pdf_path: Path, output_dir: Path, dpi: int = RENDER_DPI) -> list[Path]:
    """Renders every page of `pdf_path` to a PNG in `output_dir`, named
    `page_{page_idx:04d}.png` (0-indexed). Returns image paths in page order."""
    output_dir.mkdir(parents=True, exist_ok=True)
    zoom = dpi / 72  # PDF points are 72 per inch

    paths = []
    with fitz.open(pdf_path) as doc:
        for page_idx, page in enumerate(doc):
            pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom))
            image_path = output_dir / f"page_{page_idx:04d}.png"
            pix.save(image_path)
            paths.append(image_path)

    return paths

## Load GLM-OCR

Loaded via `AutoModelForImageTextToText` (per the model card) with `trust_remote_code=True` — GLM-OCR uses the `Glm46VForConditionalGeneration`/`Glm46VProcessor` architecture, which may not yet be in every installed `transformers` release. At 0.9B params, this is the only one of the three parser notebooks with a real CPU fallback (still slow — prefer CUDA/MPS when available).

In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"  # Apple Silicon
else:
    DEVICE = "cpu"  # 0.9B is small enough this is a real (if slow) fallback

MODEL_PATH = "zai-org/GLM-OCR"

processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16 if DEVICE != "cpu" else torch.float32,
    trust_remote_code=True,
)
model.to(DEVICE)
model.eval()

## Parse function

GLM-OCR's card documents per-task prompts (`Text Recognition:`, `Formula Recognition:`, `Table Recognition:`) rather than a single whole-page-to-Markdown prompt. To match `olmocr.ipynb`'s output contract (one Markdown string per page, tables as HTML, equations as LaTeX, original structure preserved, headers/footers dropped), we use a custom prompt adapted from olmOCR's `build_no_anchoring_v4_yaml_prompt` — this is outside GLM-OCR's officially benchmarked usage, so validate output quality on real pages before trusting it at scale.

Also unlike the Qwen-VL-based notebooks, GLM-OCR's processor takes the chat-templated messages straight through `apply_chat_template(..., tokenize=True, return_dict=True)` — no separate `qwen_vl_utils.process_vision_info` call needed.

In [ ]:
from PIL import Image

PAGE_MARKDOWN_PROMPT = (
    "Return the plain text representation of this document page as if you were reading "
    "it naturally. Convert equations to LaTeX and tables to HTML.\n"
    "Preserve the original text structure: keep headings, paragraph breaks, list items, and "
    "reading order (including multi-column layouts, read left-to-right column by column) as "
    "they appear on the page. Do not merge separate paragraphs or reflow line/paragraph breaks.\n"
    "Remove page headers and footers (e.g. running titles, page numbers), but keep references "
    "and footnotes.\n"
    "Return only the page's Markdown content, no extra commentary."
)


def parse_page(image_path: Path, max_new_tokens: int = 8000) -> str:
    """Returns the page's Markdown transcription for one page image."""
    page_image = Image.open(image_path).convert("RGB")
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": page_image},
                {"type": "text", "text": PAGE_MARKDOWN_PROMPT},
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None
        )

    generated_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    text = processor.batch_decode(
        generated_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0].strip()

    return text

## Smoke test on one PDF, one page

In [ ]:
sample_pdf = pdf_paths[0]
sample_out_dir = OUTPUT_ROOT / sample_pdf.stem / "glm_ocr"
sample_pages = render_pages(sample_pdf, sample_out_dir / "rendered_pages")

sample_text = parse_page(sample_pages[0])
print(sample_text)

## Batch: parse every PDF under `data/textbook/`

For each PDF: render pages, parse each page with GLM-OCR, concatenate per-page Markdown into one `.md` file. Skips PDFs whose output already exists, so this cell is safe to re-run after an interruption.

In [ ]:
from tqdm.auto import tqdm

PAGE_BREAK = "\n\n---\n\n"

for pdf_path in tqdm(pdf_paths, desc="PDFs"):
    out_dir = OUTPUT_ROOT / pdf_path.stem / "glm_ocr"
    out_md_path = out_dir / f"{pdf_path.stem}.md"
    if out_md_path.exists():
        continue

    page_image_paths = render_pages(pdf_path, out_dir / "rendered_pages")

    page_texts = []
    for image_path in tqdm(page_image_paths, desc=pdf_path.stem, leave=False):
        text = parse_page(image_path)
        page_texts.append(text or "")

    out_md_path.write_text(PAGE_BREAK.join(page_texts), encoding="utf-8")